# Modern Portfolio Theory, From Scratch

This notebook uses the synthetic price fallback (no network dependency), so it runs identically everywhere. Swap `allow_live=True` with normal internet access to use real prices — the API is identical either way.

In [ ]:
import sys
sys.path.insert(0, '../src')
from portfolio_lab import optimizer as opt
from portfolio_lab.data_loader import load_price_history

tickers = ['AAPL', 'MSFT', 'RELIANCE.NS', 'TCS.NS', '7203.T', '0700.HK']
result = load_price_history(tickers, '2021-01-01', '2024-01-01', allow_live=False)
print(result.status)
result.prices.tail()

In [ ]:
mu = opt.expected_returns(result.prices)
cov = opt.covariance_matrix(result.prices)
mu

## Minimum-variance portfolio (closed form via Lagrange multipliers)

`w_mv = Sigma^-1 1 / (1^T Sigma^-1 1)` — see `docs/METHODOLOGY.md` for the full derivation.

In [ ]:
w_mv = opt.min_variance_weights_closed_form(cov.values)
for t, w in zip(tickers, w_mv):
    print(f'{t:>14}: {w:+.3f}')
print('Portfolio return:', opt.portfolio_return(w_mv, mu.values))
print('Portfolio volatility:', opt.portfolio_volatility(w_mv, cov.values))

## Maximum-Sharpe (tangency) portfolio

`w_tan = Sigma^-1 (mu - rf*1) / [1^T Sigma^-1 (mu - rf*1)]`

In [ ]:
rf = 0.04
w_tan = opt.tangency_weights_closed_form(mu.values, cov.values, rf)
print('Sharpe (tangency):', opt.sharpe_ratio(w_tan, mu.values, cov.values, rf))
print('Sharpe (min-variance):', opt.sharpe_ratio(w_mv, mu.values, cov.values, rf))
assert opt.sharpe_ratio(w_tan, mu.values, cov.values, rf) >= opt.sharpe_ratio(w_mv, mu.values, cov.values, rf)

## Efficient frontier

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

r_vertex = opt.frontier_vertex_closed_form(mu.values, cov.values)
targets = np.linspace(r_vertex, mu.values.max() * 1.1, 30)
frontier = opt.efficient_frontier_closed_form(mu.values, cov.values, targets)

plt.plot(frontier['volatility'], frontier['target_return'], label='Efficient frontier')
plt.scatter([opt.portfolio_volatility(w_mv, cov.values)], [opt.portfolio_return(w_mv, mu.values)], color='blue', label='Min-variance', zorder=5)
plt.scatter([opt.portfolio_volatility(w_tan, cov.values)], [opt.portfolio_return(w_tan, mu.values)], color='red', label='Tangency', zorder=5)
plt.xlabel('Volatility')
plt.ylabel('Expected return')
plt.legend()
plt.title('Efficient frontier (synthetic demo data)')
plt.show()